# บทเรียนที่ 18: การปกป้องเอเย่นต์ AI ด้วยใบเสร็จรับเงินแบบเข้ารหัส

## สมุดบันทึกแบบลงมือทำ

สมุดบันทึกนี้จะพาคุณผ่านสี่แบบฝึกหัด:

1. **ลงนามใบเสร็จรับเงินใบแรกของคุณ** สำหรับการเรียกเครื่องมือเอเย่นต์และตรวจสอบความถูกต้อง
2. **ปลอมแปลงใบเสร็จรับเงิน** และดูว่าการตรวจสอบล้มเหลว
3. **สร้างเครือข่ายใบเสร็จรับเงินสามใบ** และยืนยันความสมบูรณ์ของเครือข่าย
4. **ห่อหุ้มการเรียกใช้เครื่องมือ Microsoft Agent Framework** เพื่อให้ทุกการกระทำสร้างใบเสร็จรับเงิน

รากฐานการเข้ารหัสทั้งหมดนำเข้าจากไลบรารีที่ได้รับการดูแลอย่างดี (`pynacl` สำหรับ Ed25519, `jcs` สำหรับ JSON กฎเกณฑ์ RFC 8785, `hashlib` จากไลบรารีมาตรฐาน Python สำหรับ SHA-256) ตรรกะใบเสร็จรับเงินเองเป็น Python ธรรมดาที่คุณสามารถอ่านและแก้ไขได้

รันเซลล์ตามลำดับแต่ละส่วนสั้นและเป็นอิสระ


## การตั้งค่า

ติดตั้ง dependencies ทั้งสอง ตัวทั้งคู่มีใบอนุญาตแบบยืดหยุ่น (Apache-2.0 / MIT)


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## เครื่องมือตัวช่วย

ตัวช่วยสองตัวนี้จัดการการเข้ารหัส base64url (โดยไม่เติมช่องว่าง) และการแฮช SHA-256 ของวัตถุใด ๆ ก็ได้ พวกมันช่วยให้สมุดบันทึกส่วนที่เหลือมุ่งเน้นไปที่ตรรกะใบเสร็จโดยตรง


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ส่วนที่ 1: ลงชื่อในใบเสร็จแรกของคุณ

ลองจินตนาการว่าเอเย่นต์ของเราสำหรับ **Contoso Travel** เพิ่งค้นหาตั๋วเครื่องบินจากซิดนีย์ไปลอสแองเจลิสให้ลูกค้า เราต้องการบันทึกคำชี้แจงของเกตเวย์เกี่ยวกับการเรียกเครื่องมือนี้เพื่อให้นักตรวจสอบในอนาคตสามารถตรวจสอบได้ภายใต้กุญแจผู้ตรวจสอบที่ปักหมุดแยกต่างหาก

### ขั้นตอนที่ 1.1: สร้างกุญแจสำหรับการลงชื่อ

ในการผลิต กุญแจสำหรับลงชื่อของเอเย่นต์จะอยู่อุปกรณ์รักษาความปลอดภัยฮาร์ดแวร์ (HSM), Azure Key Vault หรือที่เก็บข้อมูลที่ป้องกันอย่างคล้ายกัน สำหรับบทเรียนนี้เราสร้างกุญแจใหม่ในหน่วยความจำ


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### ขั้นตอนที่ 1.2: สร้างข้อมูล payload ของใบเสร็จ

ข้อมูล payload ประกอบด้วยทุกสิ่งที่เราต้องการให้ใบเสร็จรับรอง: ใครเป็นผู้ดำเนินการ, เครื่องมืออะไร, พร้อมด้วยอาร์กิวเมนต์อะไร, ผลลัพธ์ที่ได้คืออะไร, ภายใต้ข้อกำหนดใด, และเวลาใด เราจะทำการแฮชอาร์กิวเมนต์และผลลัพธ์แทนการใส่ลงไปโดยตรงเพื่อป้องกันไม่ให้ใบเสร็จเปิดเผยเนื้อหาที่ละเอียดอ่อน


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### ขั้นตอนที่ 1.3: ลงชื่อและประกอบใบเสร็จ

สามขั้นตอน:

1. ปรับ payload ให้เป็นรูปแบบ canonical โดยใช้ JCS เพื่อให้สองการดำเนินการที่สร้างใบเสร็จแบบตรรกะเดียวกันได้ไบต์ที่เหมือนกันทุกประการ
2. ลงนามกับไบต์ canonical โดยตรงโดยใช้กุญแจส่วนตัว Ed25519 PureEdDSA จะทำการแฮชข้อความภายใน ดังนั้นการแฮชเพิ่มอีกครั้งจะทำให้โปรโตคอลเปลี่ยนไป

จากนั้นลายเซ็นจะถูกแนบกับ payload ต้นฉบับเพื่อสร้างใบเสร็จขั้นสุดท้าย


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### ขั้นตอนที่ 1.4: ตรวจสอบใบเสร็จ

การตรวจสอบเป็นการย้อนกระบวนการ เราลบลายเซ็นออก คำนวณไบต์มาตรฐานอีกครั้ง และตรวจสอบลายเซ็นกับกุญแจสาธารณะของผู้ออกใบเสร็จ

การตรวจสอบทำงานแบบออฟไลน์: ไม่มีบริการที่ต้องเรียกใช้และไม่มีการเข้าถึงเครือข่าย สิ่งที่ต้องใช้ยังมีอีกอย่างนอกจากใบเสร็จ: กุญแจสาธารณะของผู้ออกใบเสร็จ ซึ่งได้มาผ่านช่องทางที่ผู้ตรวจสอบไว้วางใจแล้ว เช่นชุด JWK ที่ผู้ออกใบเสร็จเผยแพร่ หรือค่าที่ถูกกำหนดล่วงหน้า `public_key` ฟิลด์ภายในใบเสร็จไม่สามารถทำหน้าที่นั้นได้ เพราะคนที่เขียนใบเสร็จคือคนเดียวกับที่เขียนฟิลด์นั้น มาตรา 2 แสดงการปลอมแปลงที่ป้องกันได้


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


คุณควรเห็น `Receipt is valid: True`, `Pre-hashed receipt valid: False` และ `Malformed receipts rejected: True` กรณีบวกพิสูจน์ว่าเส้นทางการเซ็นชื่อแบบ direct-JCS ใช้งานได้จริง; การควบคุมเชิงลบทำให้กฎขอบเขตลายเซ็นสามารถดำเนินการได้แทนที่จะปล่อยไว้เป็นข้อความ; บรรทัดสุดท้ายแสดงอินพุตที่ไม่ถูกต้องส่งคืน `False` แทนที่จะเกิดข้อยกเว้น


## ส่วนที่ 2: ปลอมแปลงใบเสร็จ

จุดประสงค์ทั้งหมดของใบเสร็จคือต้องสามารถตรวจจับการปลอมแปลงได้ มาแสดงให้เห็นกันเถอะ

เราจะเปลี่ยนเพียงตัวอักษรเดียวในใบเสร็จและสังเกตว่าการตรวจสอบล้มเหลว


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### เกิดอะไรขึ้นเมื่อกี้?

เมื่อเราเปลี่ยน `policy_id` ไบต์แบบแคนอนิคัลก็เปลี่ยนไป ลายเซ็น (ซึ่งอยู่บนไบต์แบบแคนอนิคัลเดิม) จึงไม่ตรงกันอีกต่อไป การตรวจสอบจึงคืนค่าเป็น `False` อย่างถูกต้อง

ไม่มีทางที่จะเปลี่ยนฟิลด์ที่ลงนามแล้ว ซึ่งหมายถึงอะไรที่อยู่นอกอ็อบเจ็กต์ `signature` และยังคงให้ใบเสร็จตรวจสอบกับกุญแจสาธารณะของผู้ออกได้ เว้นแต่ผู้โจมตีจะมีคีย์ส่วนตัวของผู้ออก อ็อบเจ็กต์ `signature` เองไม่ได้ถูกลงนาม: `sig` คือ ลายเซ็น และ `verify_receipt` จะไม่สนใจ `public_key` ดังนั้นการแก้ไขฟิลด์นี้เพียงอย่างเดียวจะไม่เปลี่ยนแปลงอะไร ตราบใดที่คีย์ส่วนตัวอยู่ในที่เก็บกุญแจ และผู้ตรวจสอบใช้กุญแจที่ผู้ออกเผยแพร่อย่างอิสระ การเปลี่ยนแปลงฟิลด์ที่ลงนามแล้วเป็นไปไม่ได้ที่จะปิดบัง เงื่อนไขนี้มีความสำคัญ: เซลล์ถัดไปแสดงใบเสร็จที่ถูกแก้ไขซึ่งตรวจสอบสมบูรณ์กับกุญแจที่ต่างกัน

ลองทำเอง: แก้ไข `tool_name` หรือ `agent_id` หรือ `timestamp` ในเซลล์ข้างบนแล้วรันใหม่ การเปลี่ยนแปลงทุกอย่างในฟิลด์ที่ลงนามจะทำให้ใบเสร็จไม่ถูกต้อง


### การปลอมแปลงที่ลายเซ็นจับไม่ได้

ผู้โจมตีด้านบนไม่ได้ลงชื่อใหม่ ผู้ที่ระมัดระวังมากกว่าจะ: เปลี่ยนใบเสร็จ,
ลงชื่อด้วยกุญแจที่พวกเขาสร้างขึ้นเอง และใส่กุญแจนั้นใน
`public_key` ฟิลด์ ผลลัพธ์คือใบเสร็จที่ลายเซ็นนั้นเป็นของแท้ทั้งหมด
สำหรับเนื้อหาของมัน

รันสิ่งนี้และดูว่าการตรวจสอบสองอย่างไหนจับได้


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


คุณควรจะเห็น `True` จากนั้น `False` บรรทัดสาม `assert` นั้นทำให้เป็น
การตรวจสอบ: เซลล์จะทำให้เกิดข้อผิดพลาดหากการปลอมแปลงใดๆ ผ่านการตรวจสอบกับ
กุญแจของผู้ออกใบรับรอง

การปลอมแปลงนั้นสอดคล้องภายในจริงๆ ลายเซ็นของมันตรงกับ
เนื้อหาและกุญแจที่มันถืออยู่ ดังนั้นตัวตรวจสอบที่อ่านกุญแจจาก
ใบเสร็จนั้นจึงรายงานว่าเป็นของแท้ ไม่มีสิ่งใดในใบเสร็จดูผิดปกติ เพราะว่า
ไม่มีสิ่งใดในนั้นที่ไม่สอดคล้องกัน

สิ่งเดียวที่จับมันได้คือการตรวจสอบกับกุญแจที่ผู้โจมตีไม่มี
วิธีการที่จะมีอิทธิพล นี่คือเหตุผลที่ `verify_receipt` รับคีย์เป็นอาร์กิวเมนต์: a
ฟังก์ชันที่มีลายเซ็น `verify_receipt(receipt)` ไม่สามารถเขียนได้อย่างปลอดภัย,
เพราะว่ากุญแจเดียวที่มันสามารถเข้าถึงได้คือกุญแจที่ผู้โจมตีได้จัดหาให้

ใบเสร็จเดียวกันจะถูกบันทึกเป็น `sample_receipts/04_resigned_receipt.json` หากคุณ
ต้องการตรวจสอบนอกสมุดบันทึก


## ส่วนที่ 3: การเชื่อมใบเสร็จเข้าด้วยกัน

ใบเสร็จหนึ่งใบปกป้องการทำงานหนึ่งอย่าง ตัวแทนส่วนใหญ่ทำหลายการทำงาน เพื่อทำให้ลำดับทั้งหมดตรวจจับการปลอมแปลงได้ เราจึงเชื่อมใบเสร็จแต่ละใบกับใบเสร็จใบก่อนหน้าด้วยการใส่ค่าแฮชของใบเสร็จใบก่อนหน้าในข้อมูลของใบเสร็จใหม่

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

หากใครก็ตามลบหรือสลับตำแหน่งใบเสร็จในลำดับที่แสดง ลิงก์ถัดไปจะขาด การตรวจจับการตัดทอนหรือลำดับต่อท้ายที่ควบคุมโดยผู้เซ็นยังต้องใช้หัวปัจจุบันที่ตรึงไว้, จุดตรวจสอบภายนอก หรือการยืนยันประชากรที่ตรวจสอบได้


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Now break the chain by tampering with the middle receipt and re-verify. The tampered receipt fails its signature check, AND the next receipt fails its chain-link check (because its `previous_receipt_hash` no longer matches the modified middle receipt's hash).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


ใบเสร็จ 0 ยังคงตรวจสอบได้ (ไม่ได้ถูกแก้ไขและไม่มีใบเสร็จที่ต้องพึ่งพาก่อนหน้า). ใบเสร็จ 1 ไม่ผ่านการตรวจสอบลายเซ็นเพราะเราทำการเปลี่ยนแปลง `tool_args_hash`. ใบเสร็จ 2 ไม่ผ่านการตรวจสอบลิงก์เชนเพราะ `previous_receipt_hash` ของมันถูกคำนวณกับใบเสร็จ 1 ดั้งเดิม (ที่ตอนนี้ถูกแก้ไขแล้ว).

แม้ว่าผู้โจมตีจะลงลายมือชื่อใหม่ในใบเสร็จที่แก้ไขแล้ว 1 แต่ลายเซ็นใหม่จะล้มเหลวเมื่อเทียบกับกุญแจของผู้ออกใบเสร็จ เว้นแต่พวกเขาจะถือกุญแจส่วนตัวของผู้ออก (การลงลายมือชื่อใหม่ด้วยกุญแจของตนเองล้มเหลวเช่นเดียวกับที่แสดงในส่วนที่ 2) และความไม่ตรงกันของลิงก์โซ่ในใบเสร็จ 2 ก็ยังคงเปิดเผยการปลอมแปลง ในการซ่อนการเปลี่ยนแปลง ผู้โจมตีจะต้องลงลายมือชื่อใหม่ในทุกใบเสร็จตั้งแต่จุดที่แก้ไขเป็นต้นไป ซึ่งต้องมีการครอบครองกุญแจส่วนตัวของผู้ออก


## ส่วนที่ 4: ห่อหุ้มการเรียกเครื่องมือเอเย่นต์ด้วยการลงนามใบเสร็จ

ในการปรับใช้ในสถานการณ์จริง คุณไม่ต้องการให้นักเขียนเอเย่นต์ทุกรายจดจำที่จะต้องเรียกใช้ `make_receipt` คุณต้องการให้การลงนามใบเสร็จเป็นไปโดยอัตโนมัติสำหรับทุกการเรียกเครื่องมือ

นี่คือตัวอย่างรูปแบบที่ง่ายที่สุด: คลาสห่อหุ้มที่รับฟังก์ชันเครื่องมือที่เรียกใช้งานได้ใด ๆ และส่งคืนเวอร์ชันที่สร้างใบเสร็จรับเงิน ซึ่งสามารถปรับใช้กับกรอบเอเจนต์ใด ๆ รวมถึง Microsoft Agent Framework (`agent_framework.foundry`) ด้วย

หากคุณยังไม่มีโปรเจ็กต์ Microsoft Foundry ตั้งค่าไว้ ตัวจำลองในพื้นที่ด้านล่างนี้ยังคงสาธิตรูปแบบนี้ได้


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### การผสานรวมกับ Microsoft Agent Framework

ตัวห่อ `ReceiptedTool` ข้างต้นเป็นอิสระจากเฟรมเวิร์ก ในการใช้งานภายในเอเย่นต์ที่สร้างด้วย Microsoft Agent Framework ให้ลงทะเบียนฟังก์ชันที่ถูกห่อเป็นเครื่องมือ แบบร่าง (คุณจะต้องแทนที่ม็อกด้วยการลงทะเบียนเครื่องมือจริงของ Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

เฟรมเวิร์กเอเย่นต์ไม่จำเป็นต้องรู้เกี่ยวกับใบเสร็จ การเซ็นชื่อใบเสร็จถูกห่อรอบเครื่องมือ ไม่ได้ผนึกเข้ากับเฟรมเวิร์ก นี่คือวิธีเพิ่มต้นทางของข้อมูลให้กับโค้ดเอเย่นต์ที่มีอยู่โดยไม่ต้องเขียนโค้ดเอเย่นต์ใหม่


## สรุปและความท้าทายขยายความ

คุณได้ทำ:

- สร้างคู่คีย์ Ed25519
- สร้างและลงนามใบเสร็จสำหรับการเรียกเครื่องมือเอเย่นต์
- ตรวจสอบใบเสร็จแบบออฟไลน์โดยใช้เฉพาะกุญแจสาธารณะของผู้ออกซึ่งได้มาแยกต่างหากจากใบเสร็จ
- ปลอมแปลงใบเสร็จและสังเกตว่าการตรวจสอบล้มเหลว
- สร้างลำดับใบเสร็จที่เชื่อมโยงกันด้วยแฮชสามใบเสร็จ
- ปลอมแปลงส่วนกลางของโซ่และสังเกตทั้งความล้มเหลวของลายเซ็นและความล้มเหลวของการเชื่อมโยงโซ่
- ห่อหุ้มฟังก์ชันของเครื่องมือด้วยการลงนามใบเสร็จอัตโนมัติ

**ความท้าทายเพิ่มเติม.** ขยายสคีมารับใบเสร็จด้วย `request_id` ฟิลด์ (UUID สำหรับการติดตามแบบแจกจ่าย). อัปเดต `make_receipt` เพื่อรวมฟิลด์นี้ และยืนยันว่าใบเสร็จยังตรวจสอบได้ตั้งแต่ต้นจนจบ จากนั้นแก้ไขฟิลด์หลังการเซ็นชื่อและยืนยันว่าการตรวจสอบล้มเหลว วิธีนี้จะบังคับให้คุณเข้าใจว่าแต่ละไบต์ของการเข้ารหัสแบบ canonical มีส่วนอย่างไรกับลายเซ็น

**ขอบเขตที่สำคัญ** ภายใต้กุญแจและบริบทของเชนที่ตรึงไว้ ใบเสร็จพิสูจน์การระบุลายเซ็น ความสมบูรณ์ของเนื้อหา และลิงก์ไปยังผู้สืบทอดก่อนหน้าที่นำเสนอ พวกมันไม่ได้พิสูจน์การดำเนินการ ความจริงของแหล่งที่มา เวลาเชื่อถือได้ ความสมบูรณ์ การประเมินนโยบาย หรือความถูกต้อง ใบเสร็จเป็นพื้นฐาน การกำกับดูแลคือระบบที่คุณสร้างขึ้นด้านบน

อ่าน README ของบทเรียนอีกครั้งโดยคำนึงถึงขอบเขตนั้น ข้อผิดพลาดที่พบบ่อยที่สุดที่ทีมมักทำกับใบเสร็จคือการสมมติว่า "เรามีใบเสร็จ" หมายความว่า "เราถูกกำกับดูแล" ซึ่งไม่ใช่ ใบเสร็จทำให้พฤติกรรมของตัวแทนสามารถตรวจสอบได้ แต่ไม่ได้ทำให้พฤติกรรมนั้นถูกต้อง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
